In [9]:
import requests
import pandas as pd

def fetch_dnse_one_shot(start_date='2020-01-01', end_date='2025-12-31', resolution='1'):
    url = 'https://api.dnse.com.vn/chart-api/v2/ohlcs/derivative'
    
    # Header ẩn danh của DNSE
    headers = {
        'accept': 'application/json, text/plain, */*',
        'accept-language': 'vi-VN,vi;q=0.9',
        'origin': 'https://banggia.dnse.com.vn',
        'referer': 'https://banggia.dnse.com.vn/',
        'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }

    # 1. Chuyển đổi ngày sang Timestamp (Ép về chuẩn múi giờ VN)
    start_dt = pd.to_datetime(start_date).tz_localize('Asia/Ho_Chi_Minh')
    end_dt = pd.to_datetime(end_date + ' 23:59:59').tz_localize('Asia/Ho_Chi_Minh')

    start_ts = int(start_dt.timestamp())
    end_ts = int(end_dt.timestamp())

    print(f"🚀 ĐANG KÉO THẲNG DATA DNSE: {start_date} ĐẾN {end_date} (Khung {resolution}m)")
    print("⏳ Xin vui lòng chờ, quá trình này tải về 1 cục rất to nên sẽ mất vài giây...")

    # 2. Setup Parameter cho một lần gọi duy nhất
    params = {
        'from': start_ts,
        'to': end_ts,
        'symbol': 'VN30F1M',
        'resolution': resolution
    }

    try:
        # Bắn duy nhất 1 Request
        response = requests.get(url, headers=headers, params=params)
        
        if response.status_code == 200:
            data = response.json()
            
            # Cấu trúc của DNSE: nếu 't' có tồn tại thì bơm thẳng vào DataFrame
            if data.get('t') and len(data['t']) > 0:
                df = pd.DataFrame({
                    'Datetime': data['t'],
                    'Open': data['o'],
                    'High': data['h'],
                    'Low': data['l'],
                    'Close': data['c'],
                    'Volume': data['v']
                })
                
                print(f"✅ Đã tải về bộ nhớ {len(df):,} nến. Đang chuẩn hóa...")
                
                # 3. Chuyển đổi timestamp (giây) -> Múi giờ VN
                df['Datetime'] = pd.to_datetime(df['Datetime'], unit='s') \
                                   .dt.tz_localize('UTC') \
                                   .dt.tz_convert('Asia/Ho_Chi_Minh') \
                                   .dt.tz_localize(None)
                                   
                # Lọc trùng, sắp xếp và cắt đuôi giây
                df = df.drop_duplicates(subset=['Datetime']).sort_values('Datetime')
                df['Datetime'] = df['Datetime'].dt.floor('min')
                
                # Đưa Datetime lên làm Index
                df.set_index('Datetime', inplace=True)
                
                # 4. Xuất ra file CSV
                filename = f"data/DNSE_VN30F_{resolution}m.csv"
                df.to_csv(filename)
                
                print("=========================================")
                print(f"🎉 HOÀN TẤT TRONG 1 NỐT NHẠC!")
                print(f"📁 Dữ liệu được lưu tại: {filename}")
                print("=========================================")
                return df
            else:
                print("⚠️ Server trả về thành công nhưng mảng dữ liệu trống.")
                return None
        else:
            print(f"❌ Lỗi API: {response.status_code}")
            return None
            
    except Exception as e:
        print(f"❌ Lỗi kết nối hoặc hết bộ nhớ: {e}")
        return None

# ==========================================
# GỌI HÀM THỰC THI (CHẠY TỪ ĐÂY)
# ==========================================
if __name__ == "__main__":
    df_result = fetch_dnse_one_shot(start_date='2018-01-01', end_date='2025-12-31', resolution='1H')
    
    if df_result is not None:
        print("\n[Preview 5 dòng đầu tiên]")
        print(df_result.head())
        print("\n[Preview 5 dòng cuối cùng]")
        print(df_result.tail())

🚀 ĐANG KÉO THẲNG DATA DNSE: 2018-01-01 ĐẾN 2025-12-31 (Khung 1Hm)
⏳ Xin vui lòng chờ, quá trình này tải về 1 cục rất to nên sẽ mất vài giây...
✅ Đã tải về bộ nhớ 9,212 nến. Đang chuẩn hóa...
🎉 HOÀN TẤT TRONG 1 NỐT NHẠC!
📁 Dữ liệu được lưu tại: data/DNSE_VN30F_1Hm.csv

[Preview 5 dòng đầu tiên]
                      Open   High    Low  Close  Volume
Datetime                                               
2018-08-13 09:00:00  943.5  946.4  942.3  946.0   18959
2018-08-13 10:00:00  945.9  946.3  942.3  943.7   16381
2018-08-13 11:00:00  943.7  947.3  943.3  947.3    8416
2018-08-13 13:00:00  947.5  950.2  946.7  949.9   21499
2018-08-13 14:00:00  949.7  954.6  948.9  954.2   14072

[Preview 5 dòng cuối cùng]
                       Open    High     Low   Close  Volume
Datetime                                                   
2025-12-31 09:00:00  2021.0  2022.5  2013.8  2018.9   55557
2025-12-31 10:00:00  2018.9  2021.8  2015.5  2018.5   28695
2025-12-31 11:00:00  2018.6  2026.8  2017.4  

In [7]:
import pandas as pd

def resample_ohlcv(df, timeframe):
    """
    Hàm gộp nến (Resample) OHLCV từ khung nhỏ lên khung lớn
    :param df: DataFrame gốc (phải có index là Datetime và các cột Open, High, Low, Close, Volume)
    :param timeframe: Chuỗi thời gian Pandas hỗ trợ (VD: '5min', '10min', '15min', '30min', '1H')
    """
    
    print(f"Đang gộp nến sang khung {timeframe}...")
    
    # 1. Định nghĩa quy tắc toán học cho từng cột
    ohlcv_dict = {
        'Open': 'first',
        'High': 'max',
        'Low': 'min',
        'Close': 'last',
        'Volume': 'sum'
    }
    
    # 2. Thực hiện Resample
    # - label='left': Tên của cây nến sẽ lấy thời gian bắt đầu (Ví dụ nến 09:00 đến 09:05 sẽ tên là 09:00)
    # - closed='left': Đóng nến ở phút cuối cùng trước khi sang nến mới (Chuẩn của thị trường Việt Nam)
    df_resampled = df.resample(timeframe, label='left', closed='left').agg(ohlcv_dict)
    
    # 3. Xóa các nến rỗng (NaN)
    # Vì chứng khoán không giao dịch ban đêm và giờ nghỉ trưa, hàm resample sẽ sinh ra các dòng trống (NaN).
    # Ta phải dùng dropna() để xóa sạch các khoảng thời gian "chết" này.
    df_resampled = df_resampled.dropna()
    
    return df_resampled

In [8]:
# ==========================================
# THỰC HÀNH GỘP NẾN TỪ FILE 1 PHÚT
# ==========================================

# 1. Đọc file CSV 1 phút (Nhớ set cột Datetime làm Index ngay từ lúc đọc)
print("Đang load file dữ liệu 1 phút...")
df_1m = pd.read_csv("data\DNSE_VN30F_1m.csv", index_col='Datetime', parse_dates=True)

# 2. Gộp ra các khung thời gian tùy thích
df_3m = resample_ohlcv(df_1m, '3min')
df_3m.to_csv("DNSE_VN30F_3m.csv")

df_5m = resample_ohlcv(df_1m, '5min')
df_5m.to_csv("DNSE_VN30F_5m.csv")

df_10m = resample_ohlcv(df_1m, '10min')
df_10m.to_csv("DNSE_VN30F_10m.csv")

df_15m = resample_ohlcv(df_1m, '15min')
df_15m.to_csv("DNSE_VN30F_15m.csv")

df_30m = resample_ohlcv(df_1m, '30min')
df_30m.to_csv("DNSE_VN30F_30m.csv")

print("\n🎉 HOÀN TẤT! Đã tạo xong các file 5m, 10m, 15m, 30m.")

# In thử file 15 phút ra xem có mượt không
print("\n[Preview DataFrame 3 Phút]")
print(df_3m.head(10))

Đang load file dữ liệu 1 phút...
Đang gộp nến sang khung 3min...
Đang gộp nến sang khung 5min...
Đang gộp nến sang khung 10min...
Đang gộp nến sang khung 15min...
Đang gộp nến sang khung 30min...

🎉 HOÀN TẤT! Đã tạo xong các file 5m, 10m, 15m, 30m.

[Preview DataFrame 3 Phút]
                      Open   High    Low  Close  Volume
Datetime                                               
2018-08-13 09:00:00  943.5  943.6  942.9  943.5    1316
2018-08-13 09:03:00  943.3  943.4  942.9  943.0     839
2018-08-13 09:06:00  943.1  943.5  943.1  943.3     790
2018-08-13 09:09:00  943.2  943.3  943.1  943.2     529
2018-08-13 09:12:00  943.0  943.1  942.6  943.1     868
2018-08-13 09:15:00  943.1  943.1  942.6  942.6     597
2018-08-13 09:18:00  942.6  942.6  942.3  942.5     776
2018-08-13 09:21:00  942.5  943.2  942.4  943.0    1291
2018-08-13 09:24:00  943.1  944.2  943.1  944.2    1119
2018-08-13 09:27:00  944.4  945.9  944.4  945.3    1647


In [4]:
df_3m.tail(50)

,Open,High,Low,Close,Volume
Datetime,,,,,
2025-12-31 10:33:00,2018.0,2021.8,2017.7,2021.5,2604
2025-12-31 10:36:00,2021.2,2021.8,2018.1,2018.7,3126
2025-12-31 10:39:00,2018.6,2019.5,2018.0,2018.5,1171
2025-12-31 10:42:00,2018.4,2019.6,2018.4,2019.0,460
2025-12-31 10:45:00,2019.4,2020.1,2018.1,2019.4,1601
2025-12-31 10:48:00,2019.0,2021.3,2019.0,2019.2,1367
2025-12-31 10:51:00,2019.2,2020.0,2018.7,2019.6,648
2025-12-31 10:54:00,2019.6,2020.3,2018.6,2019.1,1016
2025-12-31 10:57:00,2019.4,2019.9,2018.5,2018.5,644
